> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTIin49w/5zM403525G-teHXho4SLHg/view?utm_content=DAGzTIin49w&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h29b78664e1)。


# 1. 环境配置

## 1.1 python 环境准备

In [2]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulsoup4==4.14.3 langchain_chroma==1.1.0 markdown==3.10 docx2txt==0.9 pymupdf==1.26.7 unstructured==0.18.26

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [3]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 代码准备

由于在向量数据库检索之前我们必须先生成一个向量数据库，因此这里我们需要将前面三节实战的内容进行整合并在该文件夹中生成一个向量数据库：

In [4]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
 chunk_size = 1500,
 chunk_overlap = 150)
splits = text_splitter.split_documents(docs)

from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os

embeddings = DashScopeEmbeddings(
  dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'))

vectordb = Chroma.from_documents(
  documents=splits,
  embedding=embeddings,
  persist_directory='./chroma')

print(vectordb._collection.count())

USER_AGENT environment variable not set, consider setting it to identify your requests.


27


# 2. 向量数据库检索

## 2.1 简介
然后我们就到了第二个阶段—提问 + 调用数据库。第一步，我们是需要根据用户提出的问题，在向量数据库里找到最相关的片段然后传给提示词。其内部原理如下：
- 首先要通过 Chorma 连接上当前的向量数据库。
- 接着将用户提问的内容用相同的 embedding 模型转为向量。
- 然后用余弦相似度计算它们的相似程度，相似度越高，代表语义越相关。
- 最后根据 Top-K 的值选择最相关的前 K 条内容进行返回。

那在 LangChain 的检索中通常有两种方式，一种是基本的相似度搜索，另外一种是进阶的 mmr 搜索：

## 2.2 相似度搜索

计算余弦相似度的方式进行检索：

In [5]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os

# 使用当前创建的向量数据库
embeddings = DashScopeEmbeddings(
 dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'), 
 model="text-embedding-v1")
vectordb = Chroma(persist_directory="./chroma", embedding_function=embeddings)

# 设置问题
question = "日常生活中的机器学习"

# 利用相似度搜索检索与问题最相关的3个切片
retriever = vectordb.as_retriever(
  search_type="similarity", 
  search_kwargs={"k": 3})
docs = retriever.invoke(question)

# 打印第一个最相关的切块内容
print(docs[0].page_content)

编写一个应用程序，接受地理信息、卫星图像和一些历史天气信息，并预测明天的天气；
编写一个应用程序，接受自然文本表示的问题，并正确回答该问题；
编写一个应用程序，接受一张图像，识别出该图像所包含的人，并在每个人周围绘制轮廓；
编写一个应用程序，向用户推荐他们可能喜欢，但在自然浏览过程中不太可能遇到的产品。

在这些情况下，即使是顶级程序员也无法提出完美的解决方案，
原因可能各不相同。有时任务可能遵循一种随着时间推移而变化的模式，我们需要程序来自动调整。
有时任务内的关系可能太复杂（比如像素和抽象类别之间的关系），需要数千或数百万次的计算。
即使人类的眼睛能毫不费力地完成这些难以提出完美解决方案的任务，这其中的计算也超出了人类意识理解范畴。
机器学习（machine learning，ML）是一类强大的可以从经验中学习的技术。
通常采用观测数据或与环境交互的形式，机器学习算法会积累更多的经验，其性能也会逐步提高。
相反，对于刚刚所说的电子商务平台，如果它一直执行相同的业务逻辑，无论积累多少经验，都不会自动提高，除非开发人员认识到问题并更新软件。
本书将带读者开启机器学习之旅，并特别关注深度学习（deep
learning，DL）的基础知识。
深度学习是一套强大的技术，它可以推动计算机视觉、自然语言处理、医疗保健和基因组学等不同领域的创新。

1.1. 日常生活中的机器学习¶
机器学习应用在日常生活中的方方面面。
现在，假设本书的作者们一起驱车去咖啡店。
阿斯顿拿起一部iPhone，对它说道：“Hey
Siri！”手机的语音识别系统就被唤醒了。
接着，李沐对Siri说道：“去星巴克咖啡店。”语音识别系统就自动触发语音转文字功能，并启动地图应用程序，
地图应用程序在启动后筛选了若干条路线，每条路线都显示了预计的通行时间……
由此可见，机器学习渗透在生活中的方方面面，在短短几秒钟的时间里，人们与智能手机的日常互动就可以涉及几种机器学习模型。
现在，假如需要我们编写程序来响应一个“唤醒词”（比如“Alexa”“小爱同学”和“Hey
Siri”）。 我们试着用一台计算机和一个代码编辑器编写代码，如
图1.1.1中所示。
问题看似很难解决：麦克风每秒钟将收集大约44000个样本，每个样本都是声波振幅的测量值。而该测量值与唤醒词难以直接关联。那又该如何编写程序，令其输入麦克风采集到的

## 2.3 最大边际相关性搜索

在向量数据库中进行信息检索时，除了相似度搜索以外，常见还有最大边际相关性（Maximum Marginal Relevance, MMR）的方法：

最大边际相关性是一种在信息检索中用于平衡 相关性 和 多样性 的技术，特别适用于需要避免重复内容并提高信息覆盖面的场景。它在传统的基于相似度的检索方法上进行了扩展，旨在通过同时考虑文档与查询的相关性以及文档之间的多样性来优化检索结果。
- 相关性（Relevance）：文档与查询之间的相似度，反映了文档对查询的相关性。
- 多样性（Diversity）：文档之间的相似度，旨在避免返回重复或冗余的内容。

In [6]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os

# 使用当前创建的向量数据库
embeddings = DashScopeEmbeddings(
 dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'), 
 model="text-embedding-v1")
vectordb = Chroma(persist_directory="./chroma", embedding_function=embeddings)

# 设置问题
question = "日常生活中的机器学习"

# 利用相似度搜索检索与问题最相关的3个切片
retriever = vectordb.as_retriever(
  search_type="mmr", 
  search_kwargs={"k": 5, "fetch_k": 10, "lambda_mult": 0.25})
docs = retriever.invoke(question)

# 打印第一个最相关的切块内容
print(docs[0].page_content)

编写一个应用程序，接受地理信息、卫星图像和一些历史天气信息，并预测明天的天气；
编写一个应用程序，接受自然文本表示的问题，并正确回答该问题；
编写一个应用程序，接受一张图像，识别出该图像所包含的人，并在每个人周围绘制轮廓；
编写一个应用程序，向用户推荐他们可能喜欢，但在自然浏览过程中不太可能遇到的产品。

在这些情况下，即使是顶级程序员也无法提出完美的解决方案，
原因可能各不相同。有时任务可能遵循一种随着时间推移而变化的模式，我们需要程序来自动调整。
有时任务内的关系可能太复杂（比如像素和抽象类别之间的关系），需要数千或数百万次的计算。
即使人类的眼睛能毫不费力地完成这些难以提出完美解决方案的任务，这其中的计算也超出了人类意识理解范畴。
机器学习（machine learning，ML）是一类强大的可以从经验中学习的技术。
通常采用观测数据或与环境交互的形式，机器学习算法会积累更多的经验，其性能也会逐步提高。
相反，对于刚刚所说的电子商务平台，如果它一直执行相同的业务逻辑，无论积累多少经验，都不会自动提高，除非开发人员认识到问题并更新软件。
本书将带读者开启机器学习之旅，并特别关注深度学习（deep
learning，DL）的基础知识。
深度学习是一套强大的技术，它可以推动计算机视觉、自然语言处理、医疗保健和基因组学等不同领域的创新。

1.1. 日常生活中的机器学习¶
机器学习应用在日常生活中的方方面面。
现在，假设本书的作者们一起驱车去咖啡店。
阿斯顿拿起一部iPhone，对它说道：“Hey
Siri！”手机的语音识别系统就被唤醒了。
接着，李沐对Siri说道：“去星巴克咖啡店。”语音识别系统就自动触发语音转文字功能，并启动地图应用程序，
地图应用程序在启动后筛选了若干条路线，每条路线都显示了预计的通行时间……
由此可见，机器学习渗透在生活中的方方面面，在短短几秒钟的时间里，人们与智能手机的日常互动就可以涉及几种机器学习模型。
现在，假如需要我们编写程序来响应一个“唤醒词”（比如“Alexa”“小爱同学”和“Hey
Siri”）。 我们试着用一台计算机和一个代码编辑器编写代码，如
图1.1.1中所示。
问题看似很难解决：麦克风每秒钟将收集大约44000个样本，每个样本都是声波振幅的测量值。而该测量值与唤醒词难以直接关联。那又该如何编写程序，令其输入麦克风采集到的

# 3. 提示词模版+大模型回复
然后我们将检索到的相关内容（上下文）与用户问题，组织成一个 Prompt（提示词）。最后交给大模型进行理解和生成回答。

- 构建提示词和模型

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 构建提示词模版
template = """请使用以下上下文信息回答最后的问题。
如果您不知道答案，就直接说您不知道，不要试图编造答案。
回答最多使用三句话。请尽可能简洁地回答。最后一定要说“谢谢提问！”。
上下文：{context}
问题：{question}
有帮助的回答："""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

llm = ChatOpenAI(model="ernie-4.0-turbo-128k",
  openai_api_key=os.environ.get("OPENAI_API_KEY"),
  base_url="https://aistudio.baidu.com/llm/lmapi/v3")

- 提问 + 回复（LCEL）

In [8]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

retriever = vectordb.as_retriever(search_type="mmr", 
 search_kwargs={"k": 1, "fetch_k": 10, "lambda_mult": 0.25})

qa_chain = (
  {"context": retriever | format_docs,
    "question": RunnablePassthrough()}
  | QA_CHAIN_PROMPT
  | llm
  | StrOutputParser())

print(qa_chain.invoke("日常生活里，哪里用到了机器学习呢？"))

日常生活中，语音识别系统（如Siri唤醒和语音转文字）、地图路线筛选、智能推荐产品以及图像中的人像识别等场景都用到机器学习。谢谢提问！


- 提问 + 回复（@chain）

In [9]:
from langchain_core.runnables import chain
@chain
def qa_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    prompt_value = QA_CHAIN_PROMPT.invoke(
        {"context": context, "question": question})
    response = llm.invoke(prompt_value)
    return StrOutputParser().invoke(response)
print(qa_chain.invoke("日常生活里，哪里用到了机器学习呢？"))

日常生活中的语音识别系统（如Siri）、地图路线推荐以及智能手机互动等场景都用到了机器学习。  
这些应用通过数据积累和模式识别自动优化性能。  
谢谢提问！


## 2.6 前端页面制作
在设计完RAG系统后，我们也来看看如何来设计前端页面：
- 从前面的流程可知，RAG系统分为两步，一步是生成向量数据库，下一步才是对话，所以在RAG的前端我们需要有上传内容的组件，并且需要通过点击按钮的方式生成我们的向量数据库。
- 然后我们就还是要设计一个聊天的页面，里面和之前一样要记录下来完整的聊天记录，并且我们也要有内容输入的页面。

In [10]:
import gradio as gr
with gr.Blocks() as demo:
  gr.Markdown('# 基础 RAG 对话平台')
  with gr.Row():
    # 创建左侧列
    with gr.Column():
      # 创建一个文本框接收网页地址
      url = gr.Textbox(label='请输入网页地址')
      url_loader_button = gr.Button('点击生成向量数据库')
      # 创建一个文件上传组件接收文件
      document = gr.File(label='请上传文件')
      document_loader_button = gr.Button('点击生成向量数据库')
      # 创建一个 Markdown 块接收数据库生成的情况
      information = gr.Markdown()

    # 创建右侧列
    with gr.Column():
      # 创建一个 Chatbot 组件记录聊天内容
      chat_history = gr.Chatbot(label='聊天记录')
      # 创建一个文本区域记录要问的问题
      input_message = gr.TextArea(label='内容输入')
      state = gr.State([])
demo.launch()

c:\Users\76391\.conda\envs\langchain_course\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


这里面有三个事件触发器，需要我们一一进行函数的补充：
- load_web_content：加载链接类的文档并载入向量数据库
- load_document：加载本地上传的文档并载入向量数据库
- chat：基于创建好的向量数据库进行对话

### 2.6.1 load_web_content()

核心功能：将传入的网页链接里的内容切分后转为向量数据库的一部分：
- 输入：url 链接
- 输出：无输出或文字信息

载入相关的库：

In [11]:
from langchain_community.document_loaders import (
  WebBaseLoader,
  YoutubeLoader)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os

设置全局变量（向量数据库创建及文本切分方法）：

In [12]:
# 初始化向量数据库
persist_directory='./chroma'
embeddings = DashScopeEmbeddings(
 dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'), 
 model="text-embedding-v1")
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
# 初始化文本切分方法
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,
 chunk_overlap=150)

向数据库里载入网页内容：

In [13]:
def load_web_content(url: str):
  # 判断内容类型
  if 'youtube.com' in url or 'youtu.be' in url:
    loader = YoutubeLoader(url)
  else:
    loader = WebBaseLoader(url)
  docs = loader.load()
  # 文本切分
  splits = text_splitter.split_documents(docs)
  vectordb = Chroma.from_documents(
    documents=splits,
    persist_directory=persist_directory,
    embedding=embeddings)
  return f"已成功在 {persist_directory} 文件夹生成向量数据库"


### 2.6.2 load_document()

核心功能：将传入的文件里的内容切分后转为向量数据库的一部分
- 输入：不同格式的文件
- 输出：无输出或文字信息

In [14]:
from langchain_community.document_loaders import (
  TextLoader,
  PyMuPDFLoader,
  Docx2txtLoader,
  UnstructuredMarkdownLoader,
  UnstructuredXMLLoader)

def load_document(file_path: str):
 # 获取文件后缀名
 file_type = file_path.split('.')[-1].lower()
 # 根据文件类型选择合适的 Loader
 if file_type == 'txt':
  loader = TextLoader(file_path, encoding="utf-8")
 elif file_type == 'pdf':
  loader = PyMuPDFLoader(file_path)
 elif file_type == 'docx':
  loader = Docx2txtLoader(file_path)
 elif file_type == 'md':
  loader = UnstructuredMarkdownLoader(file_path)
 elif file_type == 'xml':
  loader = UnstructuredXMLLoader(file_path)
 else:
  raise ValueError(f"不支持的文件类型: {file_type}")
 # 加载文档
 docs = loader.load()
 # 文本切分
 splits = text_splitter.split_documents(docs)
 vectordb = Chroma.from_documents(
  documents=splits,
  persist_directory=persist_directory,
  embedding=embeddings)
 return f"已成功在 {persist_directory} 文件夹生成向量数据库"

### 2.6.3 load_document()

核心功能：将问题和历史记录传入大模型，获得回复后将更新后的聊天记录传入聊天框中，并清空输入框的内容
- 输入：
    - 用户提问
    - 历史记录（前端 gr.State 保存）
- 输出：
    - 传入 chatbot 的历史记录
    - 更新 gr.State 的历史记录
    - 作用于输入框的空字符串

In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
def format_docs(docs):
 return "\n\n".join(doc.page_content for doc in docs)
def chat(question, chat_history):
 # 初始化聊天模型
 llm = ChatOpenAI(model="ernie-4.0-turbo-128k",
 openai_api_key=os.environ.get("OPENAI_API_KEY"),
 base_url="https://aistudio.baidu.com/llm/lmapi/v3")
 # 构建提示词模版
 template = """请使用以下上下文信息回答最后的问题。
 如果您不知道答案，就直接说您不知道，不要试图编造答案。
 回答最多使用三句话。请尽可能简洁地回答。最后一定要说“谢谢提问！”。
 上下文：{context}
 问题：{question}
 有帮助的回答："""
 QA_CHAIN_PROMPT = PromptTemplate.from_template(template)
 retriever = vectordb.as_retriever(search_type="mmr", 
 search_kwargs={"k": 1, "fetch_k": 10, "lambda_mult": 0.25})
 qa_chain = (
 {"context": retriever | format_docs,
  "question": RunnablePassthrough()}
 | QA_CHAIN_PROMPT
 | llm
 | StrOutputParser())
 result = qa_chain.invoke({question}) 
 chat_history.append({"role": "user", "content": question})
 chat_history.append({"role": "assistant", "content": result})
 return chat_history, chat_history, ""

### 2.6.4 页面整合

最后将三个函数进行整合：

In [16]:
import gradio as gr
with gr.Blocks() as demo:
  gr.Markdown('# 基础 RAG 对话平台')
  with gr.Row():
    # 创建左侧列
    with gr.Column():
      # 创建一个文本框接收网页地址
      url = gr.Textbox(label='请输入网页地址')
      url_loader_button = gr.Button('点击生成向量数据库')
      # 创建一个文件上传组件接收文件
      document = gr.File(label='请上传文件')
      document_loader_button = gr.Button('点击生成向量数据库')
      # 创建一个 Markdown 块接收数据库生成的情况
      information = gr.Markdown()
      url_loader_button.click(fn = load_web_content, inputs= url , outputs= information)
      document_loader_button.click(fn = load_document, inputs = document, outputs = information)

    # 创建右侧列
    with gr.Column():
      # 创建一个 Chatbot 组件记录聊天内容
      chat_history = gr.Chatbot(label='聊天记录')
      # 创建一个文本区域记录要问的问题
      input_message = gr.TextArea(label='内容输入')
      state = gr.State([])
      input_message.submit(fn = chat, inputs=[input_message, state],outputs=[chat_history, state, input_message])
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
